In [1]:
import sys,os
import csv
import logging
import traceback
from collections import Counter
from os.path import join as pjoin
import json

import numpy as np
from Bio import PDB
from Bio.PDB.PDBParser import PDBParser
import pandas as pd

from rna_bits.utils.data_path import get_path
from rna_bits.utils.ss import parse_ss_file
from rna_bits.utils.ss import Segmenter
from rna_bits.utils.misc import remove_string_end
from rna_bits.utils.mc_annotate import query_mc_name, mc_name_to_tuple

from rna_bits.data.benchmark.auto_from_pdb.collect import collect, collect_multi



In [2]:
# DIR = ("benchmark/auto_from_pdb/all/info/")
# df = collect(DIR)
# print(df)

DIR = ("benchmark/auto_from_pdb/all/insert_loops_sample_50_exclude_native/measures")
df = collect_multi(DIR)



In [3]:
df

rmsd        pvalue     DI_ALL   INF_ALL  \
name           instance                                                 
NR_4.0_00361.1 1         20.076946  0.000000e+00  26.638224  0.753689   
               10        19.767635  0.000000e+00  27.861148  0.709505   
               11        21.524808  3.275158e-15  30.280796  0.710840   
               12        14.691128  0.000000e+00  20.502552  0.716551   
               13        18.562496  0.000000e+00  26.929755  0.689293   
...                            ...           ...        ...       ...   
NR_4.0_99276.1 50         4.064168  1.027117e-02   6.263303  0.648886   
               6          2.415248  6.138649e-04   3.607870  0.669439   
               7          2.415248  6.138649e-04   3.607870  0.669439   
               8          2.780037  1.223993e-03   3.849282  0.722222   
               9          3.998887  9.322375e-03   6.322796  0.632456   

                           INF_WC   INF_NWC  INF_STACK  \
name           instance                                  
NR_4.0_00361.1 1         0.884678  0.500000   0.705302   
               10        0.872098  0.408248   0.652640   
               11        0.861397  0.158114   0.684737   
               12        0.907079  0.188982   0.660963   
               13        0.883208  0.333333   0.627355   
...                           ...       ...        ...   
NR_4.0_99276.1 50        0.912871 -1.000000   0.560449   
               6         1.000000 -1.000000   0.547723   
               7         1.000000 -1.000000   0.547723   
               8         0.912871 -1.000000   0.666667   
               9         0.912871 -1.000000   0.540062   

                                                                       DIR  \
name           instance                                                      
NR_4.0_00361.1 1         benchmark/auto_from_pdb/all/insert_loops_sampl...   
               10        benchmark/auto_from_pdb/all/insert_loops_sampl...   
               11        benchmark/auto_from_pdb/all/insert_loops_sampl...   
               12        benchmark/auto_from_pdb/all/insert_loops_sampl...   
               13        benchmark/auto_from_pdb/all/insert_loops_sampl...   
...                                                                    ...   
NR_4.0_99276.1 50        benchmark/auto_from_pdb/all/insert_loops_sampl...   
               6         benchmark/auto_from_pdb/all/insert_loops_sampl...   
               7         benchmark/auto_from_pdb/all/insert_loops_sampl...   
               8         benchmark/auto_from_pdb/all/insert_loops_sampl...   
               9         benchmark/auto_from_pdb/all/insert_loops_sampl...   

                                                                  REAL_DIR  
name           instance                                                     
NR_4.0_00361.1 1         /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               10        /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               11        /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               12        /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               13        /home/paul/Masters/RNA/data/benchmark/auto_fro...  
...                                                                    ...  
NR_4.0_99276.1 50        /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               6         /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               7         /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               8         /home/paul/Masters/RNA/data/benchmark/auto_fro...  
               9         /home/paul/Masters/RNA/data/benchmark/auto_fro...  

[12550 rows x 9 columns]

In [4]:
df.groupby("name").mean()

/tmp/ipykernel_172400/3755622452.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df.groupby("name").mean()


,rmsd,pvalue,DI_ALL,INF_ALL,INF_WC,INF_NWC,INF_STACK
name,,,,,,,
NR_4.0_00361.1,17.249198,2.295941e-15,23.788288,0.725279,0.874639,0.216791,0.691645
NR_4.0_00810.1,7.716808,3.001500e-04,8.863863,0.869294,0.987217,0.280000,0.808970
NR_4.0_01147.1,3.088323,0.000000e+00,3.773938,0.820407,0.938316,0.347954,0.806993
NR_4.0_01689.1,1.734442,6.343074e-06,1.931033,0.901192,1.000000,-0.540000,0.877856
NR_4.0_01806.1,12.381637,1.670234e-02,17.545775,0.707786,0.959072,-1.000000,0.615244
...,...,...,...,...,...,...,...
NR_4.0_96274.1,2.709300,0.000000e+00,3.284389,0.828878,0.933958,0.560160,0.822586
NR_4.0_97773.1,23.386576,5.451771e-01,40.566189,0.578153,0.827105,0.000000,0.570776
NR_4.0_98593.1,3.634567,0.000000e+00,4.574551,0.798191,0.909532,0.580279,0.792792


In [5]:
df.groupby("name").min()["rmsd"]

name
NR_4.0_00361.1     8.550871
NR_4.0_00810.1     4.297405
NR_4.0_01147.1     2.305917
NR_4.0_01689.1     1.240369
NR_4.0_01806.1     9.705657
                    ...    
NR_4.0_96274.1     1.959118
NR_4.0_97773.1    23.121349
NR_4.0_98593.1     1.963693
NR_4.0_98802.1    31.033798
NR_4.0_99276.1     1.316186
Name: rmsd, Length: 251, dtype: float64

In [6]:
info = collect("benchmark/auto_from_pdb/all/info/")

singles = {}
singles["from_native"] = collect_multi("benchmark/auto_from_pdb/all/insert_loops_force_native/measures")
singles["loops_top"] = collect_multi("benchmark/auto_from_pdb/all/insert_loops_top_1_exclude_native/measures")

multis =  {}
multis["loops_sample"] = collect_multi("benchmark/auto_from_pdb/all/insert_loops_sample_50_exclude_native/measures")
multis["bp2_reliable"] = collect_multi("benchmark/auto_from_pdb/bp2_limited/bp2_with_ss_reliable/measures")
multis["bp2_all"] = collect_multi("benchmark/auto_from_pdb/bp2_limited/bp2_with_ss_all/measures")


In [7]:
info

,pdb_chains,num_nucs,DIR,REAL_DIR
name,,,,
NR_4.0_00361.1,7LYG|1|A,138,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...
NR_4.0_00810.1,4C7O|1|E,48,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...
NR_4.0_01147.1,6GZ4|1|Bw,72,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...
NR_4.0_01689.1,6KYV|1|G,22,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...
NR_4.0_01806.1,3IAB|1|R,46,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...
...,...,...,...,...
NR_4.0_96870.1,7ST2|1|5,73,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...
NR_4.0_97773.1,6YDP|1|AV,67,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...
NR_4.0_98593.1,5D8H|1|A,74,benchmark/auto_from_pdb/all/info/,/home/paul/Masters/RNA/data/benchmark/auto_fro...


In [8]:
a = multis["bp2_all"]

In [9]:
a.groupby("name").min()["rmsd"]

name
NR_4.0_00810.1     6.892673
NR_4.0_01689.1     1.306609
NR_4.0_01806.1    12.691794
NR_4.0_02491.1     2.845196
NR_4.0_02858.1     2.353465
                    ...    
NR_4.0_94432.1    10.113701
NR_4.0_95171.1     3.251159
NR_4.0_95596.1     3.322353
NR_4.0_95874.1     0.674092
NR_4.0_99276.1     3.427559
Name: rmsd, Length: 127, dtype: float64

In [10]:
df = pd.DataFrame()

In [11]:
df[(

SyntaxError: unexpected EOF while parsing (3450741150.py, line 1)

In [ ]:
df[("asdf", "best")] = singles["from_native"]["rmsd"]

In [ ]:
df

In [ ]:
fn = singles["from_native"]

In [ ]:
fn[fn["rmsd"] > 2]

In [ ]:
df = pd.DataFrame([[38.0, 2.0, 18.0, 22.0, 21, np.nan],[19, 439, 6, 452, 226,232]],
                  index=pd.Index(['Tumour (Positive)', 'Non-Tumour (Negative)'], name='Actual Label:'),
                  columns=pd.MultiIndex.from_product([['Decision Tree', 'Regression', 'Random'],['Tumour', 'Non-Tumour']], names=['Model:', 'Predicted:']))
df.style



In [ ]:
df.columns

In [ ]:
df = pd.DataFrame([[38.0, 2.0, 18.0],[19, 439, 6]],
                  index=pd.Index(['Tumour (Positive)', 'Non-Tumour (Negative)'], name='Actual Label:'),
                  columns=[("adsf", "1"), ("asdf", "2"), "ddd"])
df.style



In [ ]:
pd.MultiIndex([("a", "b"), ("a", "c"), ("k", "")])

In [ ]:
info

In [ ]:
df = pd.DataFrame(index = info.index, columns = pd.MultiIndex.from_tuples([], names=["", ""]))
df[ "", "pdb_chains"] = info["pdb_chains"]

df[ "", "num_nucs"] = info["num_nucs"]

for (k,v) in singles.items():
    df[k, "."] = v.reset_index().set_index(["name"])["rmsd"]
for (k,v) in multis.items():
    df[k, "best"] = v["rmsd"].groupby("name").min()
    df[k, "mean"] = v["rmsd"].groupby("name").mean()
    df[k, "variance"] = v["rmsd"].groupby("name").var()
df.sort_values(by=[("","num_nucs")], inplace=True)
# Remove rows that have all NaN's
values = list(set(df.columns) - {("", "pdb_chains"), ("", "num_nucs")})
df.drop(df[df[values].isna().all(axis=1)].index)
df

In [19]:
def create_table(metric, best_is_min):
    df = pd.DataFrame(index = info.index, columns = pd.MultiIndex.from_tuples([], names=["", ""]))
    df[ "", "pdb_chains"] = info["pdb_chains"]

    df[ "", "num_nucs"] = info["num_nucs"]

    for (k,v) in singles.items():
        df[k, "."] = v.reset_index().set_index(["name"])[metric]
    for (k,v) in multis.items():
        if best_is_min:
            df[k, "best"] = v[metric].groupby("name").min()
        else:
            df[k, "best"] = v[metric].groupby("name").max()

        df[k, "mean"] = v[metric].groupby("name").mean()
        df[k, "variance"] = v[metric].groupby("name").var()
    df.sort_values(by=[("","num_nucs")], inplace=True)
    # Remove rows that have all NaN's
    values = list(set(df.columns) - {("", "pdb_chains"), ("", "num_nucs")})
    df.drop(df[df[values].isna().all(axis=1)].index, inplace=True)
    
    df.style.set_caption("asdfads")

    return df

# def make_pretty(styler):
#     styler.set_caption("RMSD")
#     return styler

# df = create_table("INF_ALL", best_is_min=False)
# df.style

In [20]:
with pd.ExcelWriter('output.xlsx') as writer:  
    df=create_table("rmsd", best_is_min=True)
    df.to_excel(writer, sheet_name='RMSD')
    
    df=create_table("INF_ALL", best_is_min=False)
    df.to_excel(writer, sheet_name='INF')
    
    df=create_table("DI_ALL", best_is_min=True)
    df.to_excel(writer, sheet_name='DI')


In [ ]:
values

In [18]:
df.style.to_excel("asdf.xlsx", na_rep="-")

In [16]:
df

from_native loops_top loops_sample  \
                pdb_chains num_nucs           .         .         best   
name                                                                     
NR_4.0_53827.1    1ZDJ|1|R        8    1.000000  0.912871     0.912871   
NR_4.0_11942.1    6MSF|1|S        9    0.925820  0.925820     0.925820   
NR_4.0_88374.1    4C9D|1|D        9    0.942809  0.750000     0.875000   
NR_4.0_08043.1  6Z6B|1|UUU        9    1.000000  1.000000     1.000000   
NR_4.0_41594.1    4ILM|1|C       11    0.875000  0.875000     0.875000   
...                    ...      ...         ...       ...          ...   
NR_4.0_37229.2    1MFQ|1|A      128    0.943900  0.778706     0.817866   
NR_4.0_38802.1    7LYF|1|A      135    0.896894  0.726431     0.772060   
NR_4.0_00361.1    7LYG|1|A      138    0.922741  0.698647     0.762891   
NR_4.0_47162.1    3PDR|1|X      160    0.921152  0.734461     0.725226   
NR_4.0_33922.2    4R4V|1|A      185    0.939664  0.764603     0.786098   

                                   bp2_reliable                       bp2_all  \
                    mean  variance         best      mean  variance      best   
name                                                                            
NR_4.0_53827.1  0.777443  0.017034     0.816497  0.731326  0.006758  0.816497   
NR_4.0_11942.1  0.801956  0.005442     0.771517  0.771517  0.000000  0.721688   
NR_4.0_88374.1  0.785725  0.005432     0.750000  0.750000  0.000000  0.942809   
NR_4.0_08043.1  0.757346  0.010053     0.816497  0.816497  0.000000  0.912871   
NR_4.0_41594.1  0.676380  0.007518     0.670820  0.617155  0.003184  0.707107   
...                  ...       ...          ...       ...       ...       ...   
NR_4.0_37229.2  0.738078  0.001067          NaN       NaN       NaN       NaN   
NR_4.0_38802.1  0.731988  0.000375          NaN       NaN       NaN       NaN   
NR_4.0_00361.1  0.725279  0.000436          NaN       NaN       NaN       NaN   
NR_4.0_47162.1  0.700042  0.000175          NaN       NaN       NaN       NaN   
NR_4.0_33922.2  0.757976  0.000164          NaN       NaN       NaN       NaN   

                                    
                    mean  variance  
name                                
NR_4.0_53827.1  0.647419  0.004883  
NR_4.0_11942.1  0.536936  0.014490  
NR_4.0_88374.1  0.780242  0.005139  
NR_4.0_08043.1  0.762379  0.005390  
NR_4.0_41594.1  0.652017  0.002294  
...                  ...       ...  
NR_4.0_37229.2       NaN       NaN  
NR_4.0_38802.1       NaN       NaN  
NR_4.0_00361.1       NaN       NaN  
NR_4.0_47162.1       NaN       NaN  
NR_4.0_33922.2       NaN       NaN  

[251 rows x 13 columns]